In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NAV (2).zip to NAV (2).zip


In [ ]:
import zipfile
import os

with zipfile.ZipFile('NAV (2).zip', 'r') as zip_ref:
  zip_ref.extractall('NAV_data')

In [ ]:
os.listdir('NAV_data/NAV')

['2018',
 '2024',
 '2021',
 '2022',
 '2025',
 '2017',
 '2023',
 '2016',
 '2020',
 '2015',
 '2019']

In [ ]:
import pandas as pd
import os

base_path = "NAV_data/NAV"

all_data = []

for year_folder in os.listdir(base_path):
    year_path = os.path.join(base_path, year_folder)

    if os.path.isdir(year_path):
        for file in os.listdir(year_path):
            if file.endswith(".txt"):
                file_path = os.path.join(year_path, file)

                try:
                    print(f"Reading: {file_path}")

                    df = pd.read_csv(
                        file_path,
                        sep=';',
                        skiprows=[1],
                        encoding='latin1',
                        engine = 'python',
                        on_bad_lines='skip'
                    )

                    df.columns = df.columns.str.strip()

                    #remove junk rows where scheme code is not numeric
                    df=df[df['Scheme Code'].astype(str).str.isnumeric()]

                    #print columns
                    print("Columns:", df.columns)

                    #Try flexible column selection
                    df = df.rename(columns={'Scheme Code': 'Scheme Code', 'Scheme Name': 'Scheme Name',
                                            'Net Asset Value': 'Net Asset Value', 'Net Asset Value (Rs.)': 'Net Asset Value', 'NAV': 'Net Asset Value', 'Date': 'Date'
                    })

                    #keep only required columns
                    required_cols = ['Scheme Code', 'Scheme Name', 'Net Asset Value', 'Date']
                    df = df[[col for col in required_cols if col in df.columns]]
                    all_data.append(df)

                except Exception as e:
                    print(f"Error in {file_path}: {e}")

# Combine all data
if len(all_data) == 0:
  print("No data loaded. Check file reading")
else:
  combined_df = pd.concat(all_data, ignore_index=True)

# Clean
combined_df['Date'] = pd.to_datetime(combined_df['Date'], errors='coerce')
combined_df['Net Asset Value'] = pd.to_numeric(combined_df['Net Asset Value'], errors='coerce')

combined_df = combined_df.dropna()

combined_df = combined_df.drop_duplicates(subset=['Scheme Code', 'Date'])

combined_df = combined_df.sort_values(by=['Scheme Code', 'Date'])

# Save
combined_df.to_csv("final_nav_data.csv", index=False)

print("Done!")
print("Total rows:", len(combined_df))

Reading: NAV_data/NAV/2018/02 01 Apr 18 to 29 Jun 18.txt
Columns: Index(['Scheme Code', 'Scheme Name', 'ISIN Div Payout/ISIN Growth',
       'ISIN Div Reinvestment', 'Net Asset Value', 'Repurchase Price',
       'Sale Price', 'Date'],
      dtype='object')
Reading: NAV_data/NAV/2018/04 28 Sept 18 to 26 Dec 18.txt
Columns: Index(['Scheme Code', 'Scheme Name', 'ISIN Div Payout/ISIN Growth',
       'ISIN Div Reinvestment', 'Net Asset Value', 'Repurchase Price',
       'Sale Price', 'Date'],
      dtype='object')
Reading: NAV_data/NAV/2018/01 01 Jan 18 to 31 Mar 18.txt
Columns: Index(['Scheme Code', 'Scheme Name', 'ISIN Div Payout/ISIN Growth',
       'ISIN Div Reinvestment', 'Net Asset Value', 'Repurchase Price',
       'Sale Price', 'Date'],
      dtype='object')
Reading: NAV_data/NAV/2018/05 27 Dec 18 to 31 Dec 18.txt
Columns: Index(['Scheme Code', 'Scheme Name', 'ISIN Div Payout/ISIN Growth',
       'ISIN Div Reinvestment', 'Net Asset Value', 'Repurchase Price',
       'Sale Price', 'D

In [ ]:
from google.colab import files
files.download('final_nav_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
combined_df.to_pickle('/content/drive/MyDrive/final_nav_data_new.pkl')

NameError: name 'combined_df' is not defined

In [ ]:
combined_df.head(10)

In [ ]:
# =========================================
# 🚀 IMPORTS
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from scipy.stats import ttest_ind
import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")

# =========================================
# 📂 LOAD DATA
# =========================================
df = pd.read_pickle('/content/drive/MyDrive/final_nav_data_new.pkl')

# Memory optimization
df['net_asset_value'] = df['net_asset_value'].astype('float32')
df['scheme_code'] = df['scheme_code'].astype('int32')

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['scheme_code','date'])

print("Initial shape:", df.shape)

# =========================================
# 🧠 FEATURE ENGINEERING
# =========================================

# Returns
df['return'] = df.groupby('scheme_code')['net_asset_value'].pct_change()

# Volatility (30d)
df['volatility_30'] = (
    df.groupby('scheme_code')['return']
    .rolling(30)
    .std()
    .reset_index(level=0, drop=True)
)

# Momentum (30d)
df['momentum_30'] = df.groupby('scheme_code')['net_asset_value'].pct_change(30)

# Rolling max
df['rolling_max'] = df.groupby('scheme_code')['net_asset_value'].cummax()

# Drawdown
df['drawdown'] = (df['net_asset_value'] - df['rolling_max']) / df['rolling_max']

# Drawdown flag
df['drawdown_flag'] = (df['drawdown'] <= -0.15).astype(int)

# Future NAV (252 days)
df['future_nav'] = df.groupby('scheme_code')['net_asset_value'].shift(-252)

# Peak NAV
df['peak_nav_at_drawdown'] = df['rolling_max']

# Recovery
df['recovered'] = (df['future_nav'] >= df['peak_nav_at_drawdown']).astype(int)

# Target
df['failure_flag'] = (df['recovered'] == 0).astype(int)

# Clean
df = df.dropna(subset=[
    'return','volatility_30','momentum_30','future_nav'
])

print("Final shape:", df.shape)

# =========================================
# 🔍 EDA - DATA QUALITY
# =========================================
print("\nMissing %:\n", (df.isnull().mean()*100).sort_values(ascending=False))
print("Duplicates:", df.duplicated().sum())
print("Date range:", df['date'].min(), "to", df['date'].max())

# =========================================
# 📊 EVENT ANALYSIS
# =========================================
print("Drawdown Rate:", df['drawdown_flag'].mean())
print("Failure Rate:", df['failure_flag'].mean())

# =========================================
# 📊 STATISTICAL TESTING
# =========================================
fail = df[df['failure_flag']==1]
recover = df[df['failure_flag']==0]

print("\nT-TEST RESULTS")
for col in ['drawdown','volatility_30','momentum_30','return']:
    stat, p = ttest_ind(fail[col].dropna(), recover[col].dropna())
    print(col, "p-value:", p)

# =========================================
# 📊 EFFECT SIZE
# =========================================
def cohens_d(x, y):
    return (x.mean() - y.mean()) / np.sqrt((x.var() + y.var()) / 2)

print("\nEFFECT SIZE")
for col in ['drawdown','volatility_30','momentum_30']:
    print(col, cohens_d(fail[col], recover[col]))

# =========================================
# 📊 SAMPLE FOR VISUAL EDA
# =========================================
df_sample = df.sample(100000, random_state=42)

# =========================================
# 📈 NAV TREND
# =========================================
plt.figure()
sns.lineplot(data=df_sample, x='date', y='net_asset_value')
plt.title("NAV Trend")
plt.show()

# =========================================
# 📉 DISTRIBUTION
# =========================================
sns.histplot(df_sample['return'], bins=50)
plt.title("Return Distribution")
plt.show()

# =========================================
# 📦 BOXPLOTS
# =========================================
sns.boxplot(data=df_sample, x='failure_flag', y='drawdown')
plt.title("Drawdown vs Failure")
plt.show()

sns.boxplot(data=df_sample, x='failure_flag', y='volatility_30')
plt.title("Volatility vs Failure")
plt.show()

sns.boxplot(data=df_sample, x='failure_flag', y='momentum_30')
plt.title("Momentum vs Failure")
plt.show()

# =========================================
# 🔥 FAILURE PROBABILITY TREND
# =========================================
bins = pd.qcut(df_sample['drawdown'], 5)
df_sample.groupby(bins)['failure_flag'].mean().plot(kind='bar')
plt.title("Failure vs Drawdown Levels")
plt.show()

# =========================================
# 📊 CORRELATION
# =========================================
corr = df_sample[['return','volatility_30','momentum_30','drawdown','failure_flag']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

# =========================================
# 🔥 INTERACTION EFFECT
# =========================================
pivot = df_sample.pivot_table(
    values='failure_flag',
    index=pd.qcut(df_sample['drawdown'],4),
    columns=pd.qcut(df_sample['volatility_30'],4),
    aggfunc='mean'
)

sns.heatmap(pivot, annot=True, cmap='coolwarm')
plt.title("Drawdown vs Volatility Interaction")
plt.show()

# =========================================
# ⏳ TEMPORAL ANALYSIS
# =========================================
df_sample['year'] = df_sample['date'].dt.year
df_sample.groupby('year')['failure_flag'].mean().plot()
plt.title("Failure Rate Over Time")
plt.show()

# =========================================
# 🤖 MODEL BUILDING
# =========================================
train = df[df['date'] < '2023-01-01']
test = df[df['date'] >= '2023-01-01']

features = ['return','volatility_30','momentum_30','drawdown']

X_train = train[features]
y_train = train['failure_flag']

X_test = test[features]
y_test = test['failure_flag']

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

# =========================================
# 📊 EVALUATION
# =========================================
print("\nCLASSIFICATION REPORT\n")
print(classification_report(y_test, y_pred))

roc = roc_auc_score(y_test, y_prob)
print("ROC-AUC:", roc)

# =========================================
# 🔥 FEATURE IMPORTANCE
# =========================================
importance = pd.Series(model.feature_importances_, index=features)
importance.sort_values().plot(kind='barh')
plt.title("Feature Importance")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# =========================================
# 1. TRAIN-TEST SPLIT (TIME-BASED)
# =========================================
train = df[df['date'] < '2023-01-01']
test = df[df['date'] >= '2023-01-01']

# =========================================
# 2. FEATURE SELECTION
# (Removed momentum_30 due to high correlation)
# =========================================
features = ['return', 'volatility_30', 'drawdown']

X_train = train[features]
y_train = train['failure_flag']

X_test = test[features]
y_test = test['failure_flag']

# =========================================
# 3. MODEL BUILDING
# =========================================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# =========================================
# 4. PREDICTIONS
# =========================================
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# =========================================
# 5. EVALUATION (DEFAULT THRESHOLD = 0.5)
# =========================================
print("===== DEFAULT MODEL PERFORMANCE =====")
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# =========================================
# 6. FEATURE IMPORTANCE
# =========================================
feat_imp = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(6,4))
feat_imp.plot(kind='bar', color=['#ff6b6b','#4ecdc4','#1a535c'])
plt.title("Feature Importance")
plt.ylabel("Importance Score")
plt.xticks(rotation=0)
plt.show()

# =========================================
# 7. CUSTOM THRESHOLD (BUSINESS OPTIMIZATION)
# =========================================
threshold = 0.3   # More sensitive to failures
y_pred_custom = (y_prob > threshold).astype(int)

print("\n===== CUSTOM THRESHOLD (0.3) PERFORMANCE =====")
print(classification_report(y_test, y_pred_custom))

print("\nConfusion Matrix (Custom Threshold):\n", confusion_matrix(y_test, y_pred_custom))

# =========================================
# 8. OPTIONAL: COMPARE FAILURE DETECTION
# =========================================
print("\nFailure Detection Comparison:")
print("Default Threshold (0.5) - Failures Detected:",
      confusion_matrix(y_test, y_pred)[1][1])

print("Custom Threshold (0.3) - Failures Detected:",
      confusion_matrix(y_test, y_pred_custom)[1][1])

In [ ]:
import os
os.path.exists('/content/drive/MyDrive/final_nav_data_new.pkl')

True

In [ ]:
combined_df.tail(10)

,Scheme Code,Scheme Name,Net Asset Value,Date
9475521,154308,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0260,2026-03-31
9635840,154308,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0241,2026-04-02
9635841,154308,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0294,2026-04-06
9475522,154309,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0017,2026-03-24
9475523,154309,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0045,2026-03-25
9475524,154309,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0123,2026-03-27
9475525,154309,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0229,2026-03-30
9475526,154309,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0247,2026-03-31
9635842,154309,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0232,2026-04-02
9635843,154309,HDFC CRISIL-IBX Financial Services 9-12 Months...,10.0281,2026-04-06


In [ ]:
combined_df.sample(5)

,Scheme Code,Scheme Name,Net Asset Value,Date
1870674,106169,quant Infrastructure Fund - IDCW Option - Regu...,8.6974,2018-08-10
19729583,101893,Kotak Money Market Fund - (Growth),3377.5237,2020-07-23
9965542,125306,PGIM India Midcap Fund - Regular Plan - Divide...,25.9200,2025-02-05
1515440,119293,Union Flexi Cap Fund - Direct Plan - IDCW Option,19.0100,2018-03-09
20811672,126847,Sundaram Select Micro Cap Series III - Direct ...,15.4697,2015-06-10


In [ ]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25392007 entries, 20910294 to 10455895
Data columns (total 4 columns):
 #   Column           Dtype         
---  ------           -----         
 0   Scheme Code      object        
 1   Scheme Name      object        
 2   Net Asset Value  float64       
 3   Date             datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 968.6+ MB
